In [1]:
# ── Setup — run this cell first ──────────────────────────────────────────────
# Paste the Google Drive folder ID from the shared link:
#   https://drive.google.com/drive/folders/<FOLDER_ID>
FOLDER_ID = "1rSePBwwgEaTwP2JmWi4SxMeMcYYjY8M6?usp=sharing"

import os
!pip install -q ipympl gdown
from google.colab import output
output.enable_custom_widget_manager()

DATA_DIR = "/content/strata_forced_data/"
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 8:
    import gdown
    print("Downloading data (~1.7 GB) ...")
    gdown.download_folder(id=FOLDER_ID, output=DATA_DIR, quiet=False)
    print("Download complete.")
else:
    print(f"Data already present ({len(os.listdir(DATA_DIR))} files), skipping download.")

ModuleNotFoundError: No module named 'google'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, RadioButtons, CheckButtons
%matplotlib widget

In [ ]:
## INTERACTIVE SLICE VISUALIZATIONS v2 — Colab edition

import glob, re, math
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

# ── Config ────────────────────────────────────────────────────────────────────
DS  = 8
tag = f"ds{DS}"
# DATA_DIR is set by the setup cell above
# ─────────────────────────────────────────────────────────────────────────────

_NxD, _NyD, _NzD = 4096 // DS, 2048 // DS, 4096 // DS

# ── Scan available label files and build dropdown ─────────────────────────────
_label_files = sorted(glob.glob(DATA_DIR + f"labels_{tag}_sigma*_k*.npy"))
_combos = []
for _lf in _label_files:
    _m = re.search(r'sigma(\d+)_k(\d+)\.npy', _lf)
    if _m:
        _combos.append((int(_m.group(1)), int(_m.group(2))))
if not _combos:
    raise RuntimeError("No label files found — did the setup cell run and download complete?")

_combo_dd = widgets.Dropdown(
    options=[(f"σ={s}, k={k}", (s, k)) for s, k in _combos],
    value=_combos[0],
    description="Clustering:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px"),
)
SIGMA, K = _combos[0]

# ── drdy — loaded from pre-saved DS=8 array ───────────────────────────────────
print("Loading drdy...", flush=True)
_drdy_arr = np.load(DATA_DIR + "drdy_ds8_norm.npy")  # shape (NxD, NyD, NzD), pre-normalised

def _drdy_slicer(axis, idx):
    if axis == "z":   return _drdy_arr[:, :, idx]
    elif axis == "y": return _drdy_arr[:, idx, :]
    else:             return _drdy_arr[idx, :, :]

# ── Load log_eps and log_chi ──────────────────────────────────────────────────
print("Loading log_eps...", flush=True)
_log_eps = np.load(DATA_DIR + "log_eps_ds8.npy")
print("Loading log_chi...", flush=True)
_log_chi = np.load(DATA_DIR + "log_chi_ds8.npy")

_leps_vmin = float(np.nanpercentile(_log_eps[:, :, _NzD // 2], 2))
_leps_vmax = float(np.nanpercentile(_log_eps[:, :, _NzD // 2], 98))
_lchi_vmin = float(np.nanpercentile(_log_chi[:, :, _NzD // 2], 2))
_lchi_vmax = float(np.nanpercentile(_log_chi[:, :, _NzD // 2], 98))
print("Done.", flush=True)

# ── Cache for loaded arrays ───────────────────────────────────────────────────
_bf_cache  = {}
_lbl_cache = {}

def _load_clustering(sigma, k):
    if sigma not in _bf_cache:
        print(f"Loading binary_filtered sigma={sigma}...", flush=True)
        _bf_cache[sigma] = np.load(DATA_DIR + f"binary_filtered_{tag}_sigma{sigma}.npy")
    if (sigma, k) not in _lbl_cache:
        print(f"Loading labels sigma={sigma} k={k}...", flush=True)
        _lbl_cache[(sigma, k)] = np.load(DATA_DIR + f"labels_{tag}_sigma{sigma}_k{k}.npy")
    return _bf_cache[sigma], _lbl_cache[(sigma, k)]

_bf0, _lbl0 = _load_clustering(SIGMA, K)

# ── Fields config ─────────────────────────────────────────────────────────────
FIELDS = [
    dict(path=DATA_DIR + f"r_{tag}.npy", title=r"Turbulent Density $\rho$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(slicer=_drdy_slicer, shape=(_NxD,_NyD,_NzD), title=r"$\partial \rho/\partial y + \alpha$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(data=_bf0,  title=f"Filtered overturning fraction (σ={SIGMA/DS:.4g})", cmap="RdBu_r", vmin=0, vmax=0.5),
    dict(data=_lbl0, title=f"K-means clustering (k={K})", cmap="tab10"),
    dict(data=_log_eps, title=r"$\log_{10}(\varepsilon)$", cmap="inferno", vmin=_leps_vmin, vmax=_leps_vmax),
    dict(data=_log_chi, title=r"$\log_{10}(\chi)$",        cmap="inferno", vmin=_lchi_vmin, vmax=_lchi_vmax),
]

_OVERLAY_SRC    = 3
_OVERLAY_TGTS   = list(range(len(FIELDS)))
_OVERLAY_COLORS = {0: "black", 1: "black"}

_AXIS_CFG_GEN = {
    "z": {"dim": 2, "xl": "$x$", "yl": "$y$", "T": True},
    "y": {"dim": 1, "xl": "$x$", "yl": "$z$", "T": True},
    "x": {"dim": 0, "xl": "$z$", "yl": "$y$", "T": False},
}
_gen_state = {"axis": "z", "overlay": False, "contours": [], "sigma": SIGMA, "k": K}


def _get_slice(f, axis, idx):
    if "slicer" in f: return f["slicer"](axis, idx)
    arr = f["data"]
    if axis == "x":   return arr[idx]
    elif axis == "y": return arr[:, idx]
    else:             return arr[:, :, idx]

def _field_shape(f):
    return f["shape"] if "slicer" in f else f["data"].shape

def _clear_contours():
    for co in _gen_state["contours"]: co.remove()
    _gen_state["contours"] = []

def _draw_contours(axis, idx):
    lbl_sl = _get_slice(FIELDS[_OVERLAY_SRC], axis, idx)
    k      = int(FIELDS[_OVERLAY_SRC]["data"].max()) + 1
    levels = np.arange(0.5, k)
    do_T   = _AXIS_CFG_GEN[axis]["T"]
    _ld    = lbl_sl.T if do_T else lbl_sl
    for ti in _OVERLAY_TGTS:
        co = _axs_gen[ti].contour(_ld, levels=levels,
                                   colors=_OVERLAY_COLORS.get(ti, "white"),
                                   linewidths=0.6, alpha=0.7)
        _gen_state["contours"].append(co)

def _update_label_colormap(k):
    _tab10   = plt.get_cmap("tab10").colors
    new_cmap = ListedColormap(_tab10[:k])
    _ims_gen[_OVERLAY_SRC].set_cmap(new_cmap)
    _cbars[_OVERLAY_SRC].set_ticks(list(range(k)))
    FIELDS[_OVERLAY_SRC]["vmin"] = -0.5
    FIELDS[_OVERLAY_SRC]["vmax"] = k - 0.5
    _ims_gen[_OVERLAY_SRC].set_clim(-0.5, k - 0.5)

def _update_sigma_circle(sigma):
    global _sigma_circle, _sigma_text
    _sigma_circle.remove()
    _sigma_text.remove()
    sd  = sigma / DS
    pad = sd * 0.4
    _sigma_circle = plt.Circle((sd + pad, sd + pad), sd,
                                fill=False, edgecolor='green', linewidth=3, zorder=5)
    _axs_gen[_SIGMA_PANEL].add_patch(_sigma_circle)
    _sigma_text = _axs_gen[_SIGMA_PANEL].text(sd + pad, sd + pad, r'$\sigma$',
                                               color='green', ha='center', va='center',
                                               fontsize=7, zorder=6)

# ── Build figure ──────────────────────────────────────────────────────────────
print("Building figure...", flush=True)
for _f in FIELDS:
    if "slicer" not in _f and "path" in _f:
        print(f"  {_f['path'].split('/')[-1]}", flush=True)
        _f["data"] = np.load(_f["path"])
    if "data" in _f and "vmin" not in _f and _f["data"].dtype.kind in ("i", "u"):
        _n = int(_f["data"].max()) + 1
        _f["vmin"] = -0.5
        _f["vmax"] = _n - 0.5
        _f["cmap"] = ListedColormap(plt.get_cmap("tab10").colors[:_n])
        _f["ticks"] = list(range(_n))

_PANEL_W  = 4.5
_N_COLS   = 3
_N_FIELDS = len(FIELDS)
_N_ROWS   = math.ceil(_N_FIELDS / _N_COLS)
_sl0      = _get_slice(FIELDS[0], "z", 0)
_h0, _w0  = _sl0.shape
_PANEL_H  = _PANEL_W * (_w0 / _h0)

fig_gen, _axs2d = plt.subplots(_N_ROWS, _N_COLS,
                                figsize=(_PANEL_W * _N_COLS, _PANEL_H * _N_ROWS + 0.9),
                                squeeze=False)
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.12, top=0.93,
                    wspace=0.1, hspace=0.1)

_axs_gen = [_axs2d[r][c] for r in range(_N_ROWS) for c in range(_N_COLS)]
for _ax in _axs_gen[_N_FIELDS:]: _ax.set_visible(False)
_axs_gen = _axs_gen[:_N_FIELDS]

_ims_gen, _cbars = [], []
for _ax, _f in zip(_axs_gen, FIELDS):
    _sl = _get_slice(_f, "z", 0)
    _kw = dict(origin="lower", aspect="equal", cmap=_f.get("cmap", "viridis"), interpolation="nearest")
    if "vmin" in _f: _kw["vmin"] = _f["vmin"]
    if "vmax" in _f: _kw["vmax"] = _f["vmax"]
    _im = _ax.imshow(np.ma.masked_invalid(_sl.T), **_kw)
    _cb = plt.colorbar(_im, ax=_ax, shrink=0.8, ticks=_f.get("ticks"))
    _ax.set_title(_f.get("title", ""), fontsize=9)
    _ims_gen.append(_im)
    _cbars.append(_cb)

_SIGMA_PANEL  = 2
_sd           = SIGMA / DS
_sigma_circle = plt.Circle((_sd * 1.4, _sd * 1.4), _sd,
                             fill=False, edgecolor='green', linewidth=3, zorder=5)
_axs_gen[_SIGMA_PANEL].add_patch(_sigma_circle)
_sigma_text   = _axs_gen[_SIGMA_PANEL].text(_sd * 1.4, _sd * 1.4, r'$\sigma$',
                                              color='green', ha='center', va='center',
                                              fontsize=7, zorder=6)


def _apply_gen(axis, idx):
    cfg  = _AXIS_CFG_GEN[axis]
    do_T = cfg["T"]
    for _im, _f in zip(_ims_gen, FIELDS):
        _sl  = _get_slice(_f, axis, idx)
        h, w = _sl.shape
        if do_T:
            _data = np.ma.masked_invalid(_sl.T)
            _ext  = [-0.5, h - 0.5, -0.5, w - 0.5]
            _xlim, _ylim = h - 0.5, w - 0.5
        else:
            _data = np.ma.masked_invalid(_sl)
            _ext  = [-0.5, w - 0.5, -0.5, h - 0.5]
            _xlim, _ylim = w - 0.5, h - 0.5
        if "vmin" not in _f:
            _im.set_clim(float(np.nanmin(_sl)), float(np.nanmax(_sl)))
        _im.set_data(_data)
        _im.set_extent(_ext)
        _im.axes.set_xlim(-0.5, _xlim)
        _im.axes.set_ylim(-0.5, _ylim)
        _im.axes.set_xlabel(cfg["xl"])
        _im.axes.set_ylabel(cfg["yl"])
    _clear_contours()
    if _gen_state["overlay"]:
        _draw_contours(axis, idx)
    fig_gen.suptitle(f"{axis}-slice {idx}  "
                     f"(σ={_gen_state['sigma']}, k={_gen_state['k']}, every {DS} grid pts)",
                     fontsize=11)
    fig_gen.canvas.draw_idle()


_apply_gen("z", 0)

_ax_sl_gen  = fig_gen.add_axes([0.08, 0.04, 0.60, 0.025])
_slider_gen = Slider(_ax_sl_gen, "Slice", 0, _field_shape(FIELDS[0])[2] - 1, valinit=0, valstep=1)
_ax_rb_gen  = fig_gen.add_axes([0.82, 0.01, 0.05, 0.09])
_radio_gen  = RadioButtons(_ax_rb_gen, ("z", "y", "x"), active=0)
_ax_cb_gen  = fig_gen.add_axes([0.72, 0.02, 0.08, 0.06])
_check_gen  = CheckButtons(_ax_cb_gen, ["borders"], [False])


def _on_sl_gen(val):
    _apply_gen(_gen_state["axis"], int(_slider_gen.val))

def _on_rb_gen(label):
    _gen_state["axis"] = label
    cfg = _AXIS_CFG_GEN[label]
    n   = _field_shape(FIELDS[0])[cfg["dim"]]
    _slider_gen.valmax = n - 1
    _slider_gen.ax.set_xlim(0, n - 1)
    _sl = _get_slice(FIELDS[0], label, 0)
    h, w = _sl.shape
    panel_h = _PANEL_W * (w / h) if cfg["T"] else _PANEL_W * (h / w)
    fig_gen.set_size_inches(_PANEL_W * _N_COLS, panel_h * _N_ROWS + 0.9)
    plt.subplots_adjust(bottom=0.12)
    _slider_gen.set_val(min(int(_slider_gen.val), n - 1))

def _on_check_gen(label):
    _gen_state["overlay"] = not _gen_state["overlay"]
    _apply_gen(_gen_state["axis"], int(_slider_gen.val))

def _on_dropdown(change):
    if change["name"] != "value": return
    sigma, k = change["new"]
    bf, lbl  = _load_clustering(sigma, k)
    FIELDS[2]["data"]  = bf
    FIELDS[2]["title"] = f"Filtered overturning fraction (σ={sigma/DS:.4g})"
    _axs_gen[2].set_title(FIELDS[2]["title"], fontsize=9)
    FIELDS[3]["data"]  = lbl
    FIELDS[3]["title"] = f"K-means clustering (k={k})"
    _axs_gen[3].set_title(FIELDS[3]["title"], fontsize=9)
    _gen_state["sigma"] = sigma
    _gen_state["k"]     = k
    _update_label_colormap(k)
    _update_sigma_circle(sigma)
    _apply_gen(_gen_state["axis"], int(_slider_gen.val))


_combo_dd.observe(_on_dropdown, names="value")
_slider_gen.on_changed(_on_sl_gen)
_radio_gen.on_clicked(_on_rb_gen)
_check_gen.on_clicked(_on_check_gen)

display(_combo_dd)
plt.show()